In [ ]:
# Importing all the required Libraries
import os
import torch
import cv2
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
)
import torchvision.models as models
import albumentations as A
from albumentations import Compose, HorizontalFlip, VerticalFlip, Rotate, MotionBlur, GaussianBlur, Resize
from tqdm import tqdm
import pandas as pd
import copy
import torch.nn as nn



In [ ]:
# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
NUM_EPOCHS = 10
KFOLD_SPLITS = 10
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
# Mouting Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Refering to the directory where dataset is stored in my google drive (it vary for you)
leaf_input = "/content/drive/MyDrive/datasets/Mango_Leaf_Dataset"


In [ ]:
#  Novel Class-wise augmentations
class_specific_transforms = {
    "Anthracnose": A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=25, p=0.8),
        A.MotionBlur(blur_limit=3, p=0.2),
    ]),
    "Bacterial_Black_Spot": A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=10, p=0.7),
    ]),
    "Healthy": A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Rotate(limit=10, p=0.5),
        A.RandomBrightnessContrast(p=0.1),
    ]),
    "Multiple": A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=30, p=0.8),
        A.MotionBlur(p=0.3)
    ]),
}

val_test_transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])

def wrap_transform(transform):
    return lambda image: transform(image=image)["image"]

wrapped_class_specific_transforms = {
    cls: wrap_transform(tfm) for cls, tfm in class_specific_transforms.items()
}

wrapped_val_test_transform = wrap_transform(val_test_transform)


/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
# Defining the class

class LeafDataset(Dataset):
    def __init__(self, root_dir, transform_dict=None):
        self.root_dir = root_dir
        self.transform_dict = transform_dict
        self.image_paths = []
        self.labels = []
        self.class_names = sorted(os.listdir(root_dir))

        for label_idx, class_name in enumerate(self.class_names):
            class_folder = os.path.join(root_dir, class_name)
            for img_file in os.listdir(class_folder):
                self.image_paths.append(os.path.join(class_folder, img_file))
                self.labels.append(label_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        class_name = self.class_names[label]

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform_dict and class_name in self.transform_dict:
            image = self.transform_dict[class_name](image)
        else:
            image = A.Resize(224, 224)(image=image)["image"]

        image = torch.tensor(image / 255.0, dtype=torch.float32).permute(2, 0, 1)
        return image, label


In [ ]:
# Defining a function to dynamically use the models

def get_model(model_name="resnet18", num_classes=4, pretrained=True):
    if model_name == "resnet18":
        model = models.resnet18(pretrained=pretrained)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "resnet50":
        model = models.resnet50(pretrained=pretrained)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(pretrained=pretrained)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(pretrained=pretrained)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "convnext_tiny":
        model = models.convnext_tiny(pretrained=pretrained)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)

    else:
        raise ValueError("Unsupported model name!")

    return model


In [ ]:
def run_kfold_training(
    model_name,
    dataset,
    transform_dict_train,
    transform_val,
    num_classes=4,
    batch_size=32,
    num_epochs=10,
    k_folds=5,
    save_prefix="model",
    lr=1e-4,
    seed=42,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
):
    from torch.utils.data import Subset, DataLoader
    from sklearn.model_selection import KFold
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
    )
    import pandas as pd
    import copy
    import torch.nn as nn
    import torch.optim as optim
    from tqdm import tqdm
    import torch

    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=seed)
    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
        print(f"\n🟩 Fold {fold + 1}/{k_folds}")

        train_subset = Subset(dataset, train_idx)
        val_subset = Subset(dataset, val_idx)

        # Apply transforms
        train_subset.dataset.transform_dict = transform_dict_train
        val_subset.dataset.transform_dict = {
            cls: transform_val for cls in dataset.class_names
        }

        # DataLoaders
        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

        # Model
        model = get_model(model_name, num_classes=num_classes).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
        criterion = nn.CrossEntropyLoss()

        best_val_acc = 0
        best_model_wts = copy.deepcopy(model.state_dict())

        for epoch in range(num_epochs):
            print(f"\n📚 Epoch {epoch+1}")
            model.train()
            train_loss = 0

            for images, labels in tqdm(train_loader):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            scheduler.step()

            # 🔍 Validation
            model.eval()
            all_preds, all_labels = [], []
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    preds = torch.argmax(outputs, dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            acc = accuracy_score(all_labels, all_preds)
            prec = precision_score(all_labels, all_preds, average='macro')
            rec = recall_score(all_labels, all_preds, average='macro')
            f1 = f1_score(all_labels, all_preds, average='macro')
            ber = 1 - balanced_accuracy_score(all_labels, all_preds)
            kappa = cohen_kappa_score(all_labels, all_preds)
            mcc = matthews_corrcoef(all_labels, all_preds)

            print(f"✅ Fold {fold+1} Epoch {epoch+1} — Acc: {acc:.4f}, F1: {f1:.4f}, BER: {ber:.4f}")

            if acc > best_val_acc:
                best_val_acc = acc
                best_model_wts = copy.deepcopy(model.state_dict())

        # Save best model
        torch.save(best_model_wts, f"{save_prefix}_fold{fold+1}.pth")

        # Store metrics
        fold_metrics.append({
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "BER": ber,
            "Kappa": kappa,
            "MCC": mcc
        })

    # Convert to DataFrame
    df_metrics = pd.DataFrame(fold_metrics)

    print("\n📈 K-Fold Results Summary:")
    print(df_metrics)

    print(f"\n📊 Mean ± Std across folds for {save_prefix}:")
    for metric in ["Accuracy", "Precision", "Recall", "F1 Score", "BER", "Kappa", "MCC"]:
        mean = df_metrics[metric].mean()
        std = df_metrics[metric].std()
        print(f"{metric}: {mean:.4f} ± {std:.4f}")

    return df_metrics


In [ ]:
df_ResNet50_leaf = run_kfold_training(
    model_name="ResNet_50",
    dataset=LeafDataset(root_dir=leaf_input, transform_dict=wrapped_class_specific_transforms),
    transform_dict_train=wrapped_class_specific_transforms,
    transform_val=wrapped_val_test_transform,
    num_classes=4,
    batch_size=32,
    num_epochs=10,
    k_folds=10,
    save_prefix="ResNet_50"
)



🟩 Fold 1/10


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 184MB/s]



📚 Epoch 1


100%|██████████| 67/67 [30:22<00:00, 27.20s/it]


✅ Fold 1 Epoch 1 — Acc: 0.9703, F1: 0.9627, BER: 0.0309

📚 Epoch 2


100%|██████████| 67/67 [01:24<00:00,  1.27s/it]


... (211 outputs left out)

100%|██████████| 67/67 [01:26<00:00,  1.30s/it]


✅ Fold 10 Epoch 9 — Acc: 0.9872, F1: 0.9814, BER: 0.0227

📚 Epoch 10


100%|██████████| 67/67 [01:27<00:00,  1.30s/it]


✅ Fold 10 Epoch 10 — Acc: 0.9872, F1: 0.9814, BER: 0.0227


In [ ]:
#  Resnet Model KFold Result


📊 Mean ± Std across folds on ResNet 50:
Accuracy: 0.9860 ± 0.0083
Precision: 0.9825 ± 0.0096
Recall: 0.9806 ± 0.0145
F1 Score: 0.9811 ± 0.0114
BER: 0.0194 ± 0.0145
Kappa: 0.9762 ± 0.0135
MCC: 0.9763 ± 0.0134


In [ ]:
df_mobilenet_leaf = run_kfold_training(
    model_name="mobilenet_v2",
    dataset=LeafDataset(root_dir=leaf_input, transform_dict=wrapped_class_specific_transforms),
    transform_dict_train=wrapped_class_specific_transforms,
    transform_val=wrapped_val_test_transform,
    num_classes=4,
    batch_size=32,
    num_epochs=10,
    k_folds=10,
    save_prefix="mobilenet_v2_leaf"
)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth



🟩 Fold 1/10


100%|██████████| 13.6M/13.6M [00:00<00:00, 69.5MB/s]



📚 Epoch 1


100%|██████████| 66/66 [20:05<00:00, 18.26s/it]


✅ Fold 1 Epoch 1 — Acc: 0.9745, F1: 0.9752, BER: 0.0281

📚 Epoch 2


... (196 outputs left out)

100%|██████████| 66/66 [01:08<00:00,  1.04s/it]


✅ Fold 10 Epoch 9 — Acc: 0.9957, F1: 0.9911, BER: 0.0076

📚 Epoch 10


100%|██████████| 66/66 [01:08<00:00,  1.04s/it]


✅ Fold 10 Epoch 10 — Acc: 0.9957, F1: 0.9954, BER: 0.0076

📈 K-Fold Results Summary:
   Fold  Accuracy  Precision    Recall  F1 Score       BER     Kappa       MCC
0     1  0.978723   0.974130  0.970103  0.971779  0.029897  0.963334  0.963559
1     2  0.974468   0.968907  0.959936  0.964322  0.040064  0.957738  0.957841
2     3  0.974468   0.979240  0.964364  0.971344  0.035636  0.957647  0.957946
3     4  0.982979   0.967262  0.987328  0.977020  0.012672  0.971055  0.971282
4     5  0.987234   0.981071  0.981071  0.981071  0.018929  0.976742  0.976742
5     6  0.982979   0.978944  0.972973  0.974799  0.027027  0.971765  0.972150
6     7  0.970085   0.957762  0.960299  0.958894  0.039701  0.952685  0.952712
7     8  0.995726   0.998344  0.991379  0.994783  0.008621  0.992132  0.992174
8     9  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000
9    10  0.995726   0.998408  0.992424  0.995355  0.007576  0.991699  0.991744

📊 Mean ± Std across folds for mobilenet_v2_le

In [ ]:
df_efficientnet_leaf = run_kfold_training(
    model_name="efficientnet_b0",
    dataset=LeafDataset(root_dir=leaf_input, transform_dict=wrapped_class_specific_transforms),
    transform_dict_train=wrapped_class_specific_transforms,
    transform_val=wrapped_val_test_transform,
    num_classes=4,
    batch_size=32,
    num_epochs=10,
    k_folds=10,
    save_prefix="efficientnet_b0"
)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth



🟩 Fold 1/10


100%|██████████| 20.5M/20.5M [00:00<00:00, 185MB/s]



📚 Epoch 1


100%|██████████| 67/67 [18:49<00:00, 16.86s/it]


✅ Fold 1 Epoch 1 — Acc: 0.9195, F1: 0.8854, BER: 0.1060

📚 Epoch 2


... (204 outputs left out)

100%|██████████| 67/67 [01:09<00:00,  1.04s/it]


✅ Fold 10 Epoch 9 — Acc: 0.9830, F1: 0.9825, BER: 0.0064

📚 Epoch 10


100%|██████████| 67/67 [01:10<00:00,  1.05s/it]


✅ Fold 10 Epoch 10 — Acc: 0.9872, F1: 0.9818, BER: 0.0167

📈 K-Fold Results Summary:
   Fold  Accuracy  Precision    Recall  F1 Score       BER     Kappa       MCC
0     1  0.983051   0.974802  0.981194  0.977121  0.018806  0.970827  0.971098
1     2  0.970339   0.972839  0.959687  0.966030  0.040313  0.950851  0.951086
2     3  0.978814   0.971577  0.971543  0.971247  0.028457  0.963977  0.964069
3     4  0.978723   0.960800  0.968763  0.963756  0.031237  0.964646  0.964912
4     5  0.991489   0.983007  0.989405  0.986137  0.010595  0.984556  0.984596
5     6  0.970213   0.953598  0.956509  0.953985  0.043491  0.950584  0.950959
6     7  0.978723   0.964809  0.969337  0.966951  0.030663  0.966056  0.966084
7     8  0.991489   0.989989  0.989989  0.989989  0.010011  0.984593  0.984593
8     9  0.970213   0.947850  0.982417  0.963925  0.017583  0.950198  0.951153
9    10  0.987234   0.980595  0.983256  0.981767  0.016744  0.975245  0.975279

📊 Mean ± Std across folds for efficientnet_b0

In [ ]:
df_convNext_leaf = run_kfold_training(
    model_name="convnext_tiny",
    dataset=LeafDataset(root_dir=leaf_input, transform_dict=wrapped_class_specific_transforms),
    transform_dict_train=wrapped_class_specific_transforms,
    transform_val=wrapped_val_test_transform,
    num_classes=4,
    batch_size=32,
    num_epochs=10,
    k_folds=10,
    save_prefix="convnext_tiny"
)



🟩 Fold 1/10


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1`. You can also use `weights=ConvNeXt_Tiny_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth
100%|██████████| 109M/109M [00:00<00:00, 127MB/s] 



📚 Epoch 1


100%|██████████| 67/67 [35:05<00:00, 31.42s/it]


✅ Fold 1 Epoch 1 — Acc: 0.9915, F1: 0.9891, BER: 0.0114

📚 Epoch 2


100%|██████████| 67/67 [01:28<00:00,  1.33s/it]


... (211 outputs left out)

100%|██████████| 67/67 [01:30<00:00,  1.36s/it]


✅ Fold 10 Epoch 9 — Acc: 0.9957, F1: 0.9911, BER: 0.0076

📚 Epoch 10


100%|██████████| 67/67 [01:31<00:00,  1.37s/it]


✅ Fold 10 Epoch 10 — Acc: 0.9957, F1: 0.9911, BER: 0.0076

📈 K-Fold Results Summary:
   Fold  Accuracy  Precision    Recall  F1 Score       BER     Kappa       MCC
0     1  0.983051   0.978255  0.978255  0.978255  0.021745  0.970787  0.970787
1     2  0.987288   0.989771  0.980769  0.985175  0.019231  0.978996  0.979100
2     3  0.983051   0.988176  0.973151  0.980207  0.026849  0.970983  0.971298
3     4  0.987234   0.982558  0.977941  0.979425  0.022059  0.978705  0.978971
4     5  0.991489   0.983007  0.989405  0.986137  0.010595  0.984556  0.984596
5     6  0.978723   0.971369  0.970397  0.970059  0.029603  0.964596  0.964857
6     7  0.982979   0.970543  0.976690  0.973359  0.023310  0.972855  0.972940
7     8  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000
8     9  1.000000   1.000000  1.000000  1.000000  0.000000  1.000000  1.000000
9    10  0.995745   0.990000  0.992424  0.991052  0.007576  0.991748  0.991783

📊 Mean ± Std across folds for convnext_tiny:
